# Kaggle Agent Ensemble: yash9439 + Agent Extra Tasks

Blended submission: all 128 correct solutions from the public kernel
[yash9439/neurogolf-emsembling](https://www.kaggle.com/code/yash9439/neurogolf-emsembling)
plus 4 additional tasks produced by our autonomous agent.

# Extra 4 Tasks — Agent Solutions

These 4 tasks were produced by our autonomous agent and are not covered (or incorrectly solved) by the public `yash9439/neurogolf-emsembling` kernel. They are loaded from the `jiweiliu/neurogolf-extra-4` dataset.

| Task | I/O Shape | Local Score | Strategy |
|------|-----------|-------------|----------|
| task003 | 6×3 → 9×3 | 14.25 pts | Period-based row tiling |
| task010 | 9×9 → 9×9 | 13.34 pts | Row-color histogram routing |
| task049 | 10×10 → 3×3 | 13.11 pts | Min-count color bbox extraction |
| task057 | 8×8 → 3×6 | 13.25 pts | Bbox extract + horizontal tile ×2 |

## task003 — Period-based row tiling (14.25 pts)
Input is a 6×3 grid with a repeating 3-row pattern. The network uses `Slice` to extract the 3-row base period, then `Concat` and `Pad` to tile it into the 9×3 output. The output/input mask is applied via a learned channel selector.

## task010 — Row-color histogram routing (13.34 pts)
Input is a 9×9 grid. The network detects which color channels are present per row using `Greater`/`Cast` comparisons on channel sums, then routes color values to the correct output positions via a series of `Add`/`Mul` accumulations. Uses 6 color-channel constants (`_rch_c1..c6`).

## task049 — Min-count color bbox extraction (13.11 pts)
Input is a 10×10 grid. The network counts cells per color channel (`ReduceSum`), finds the rarest color via `ArgMax` on negated counts, extracts that color's bounding box using `Gather`/`Reshape`, and outputs a tight 3×3 crop centered on the rarest color's region.

## task057 — Bbox extract + horizontal tile ×2 (13.25 pts)
Input is an 8×8 grid. The network finds the bounding box of the non-background region using `ReduceMax`/`ArgMax` on rows and columns, extracts a 3×3 subgrid via `Gather`, and tiles it twice horizontally with `Pad` to produce the 3×6 output.

---

> This solution was produced end-to-end by a fully autonomous AI agent —
> from competition discovery through solver development, optimization, and submission packaging.
> The agent was designed and developed by **Jiwei Liu**.
> Full agent session trace: [https://daxiongshu.github.io/kaggle-agent-session-example/](https://daxiongshu.github.io/kaggle-agent-session-example/)

In [ ]:
import os, zipfile

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

# yash9439 tasks that fail correctness — skip these
SKIP = {
    'task003.onnx', 'task024.onnx', 'task031.onnx', 'task057.onnx',
    'task073.onnx', 'task081.onnx', 'task082.onnx', 'task085.onnx',
    'task103.onnx', 'task113.onnx', 'task114.onnx', 'task115.onnx',
    'task120.onnx', 'task122.onnx', 'task126.onnx', 'task127.onnx',
    'task151.onnx', 'task166.onnx', 'task171.onnx', 'task176.onnx',
    'task177.onnx', 'task178.onnx', 'task188.onnx', 'task218.onnx',
    'task230.onnx', 'task232.onnx', 'task252.onnx', 'task258.onnx',
    'task266.onnx', 'task278.onnx', 'task283.onnx', 'task294.onnx',
    'task298.onnx', 'task299.onnx', 'task304.onnx', 'task305.onnx',
    'task315.onnx', 'task329.onnx', 'task331.onnx', 'task352.onnx',
    'task360.onnx', 'task375.onnx', 'task377.onnx', 'task384.onnx',
}

files = {}

# Load yash9439's correct solutions
yash_zip = '/kaggle/input/notebooks/yash9439/neurogolf-emsembling/submission.zip'
with zipfile.ZipFile(yash_zip) as z:
    for fname in sorted(z.namelist()):
        if not fname.endswith('.onnx') or fname in SKIP:
            continue
        files[fname] = z.read(fname)

print(f"Loaded {len(files)} tasks from yash9439/neurogolf-emsembling")

# Load our 4 extra tasks (agent-produced, override any overlap)
EXTRA_DIR = '/kaggle/input/datasets/jiweiliu/neurogolf-extra-4'
extra_tasks = ['task003.onnx', 'task010.onnx', 'task049.onnx', 'task057.onnx']
for fname in extra_tasks:
    path = os.path.join(EXTRA_DIR, fname)
    with open(path, 'rb') as f:
        files[fname] = f.read()
    print(f"  + {fname} (agent)")

print(f"\nTotal: {len(files)} tasks")

# Package submission.zip
zip_path = f'{OUT_DIR}/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname, data in sorted(files.items()):
        zf.writestr(fname, data)

size_mb = os.path.getsize(zip_path) / 1e6
print(f"submission.zip: {len(files)} tasks ({size_mb:.1f} MB)")